### Setup of timesfm library.
- pip installation of requirements
- Then restart the runtime.

In [1]:
# Clone the repository
!git clone https://github.com/google-research/timesfm.git
%cd timesfm

# Append the finetuning package to the packages list in pyproject.toml.
# This command finds the line starting with "packages = [" and replaces the trailing "]"
# with ', { include = "finetuning", from = "src" }]' to add the finetuning package.
!sed -i '/^packages = \[/ s/]/, { include = "finetuning", from = "src" }]/' pyproject.toml

# Install the package in editable mode
!pip install -e .

Cloning into 'timesfm'...
remote: Enumerating objects: 856, done.
remote: Counting objects: 100% (372/372), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 856 (delta 278), reused 231 (delta 226), pack-reused 484 (from 3)
Receiving objects: 100% (856/856), 2.40 MiB | 5.47 MiB/s, done.
Resolving deltas: 100% (438/438), done.
/content/timesfm
Obtaining file:///content/timesfm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
  Building editable for timesfm (pyproject.toml) ... done
  Created wheel for timesfm: filename=timesfm-1.2.8-py3-none-any.whl size=10041 sha256=eb8dae3969f5185986bc7ecb09ddf804cc39c690f6f861e44291b43e3adf7371
  Stored in directory: /tm

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.0 MB/s eta 0:00:00


In [3]:
!pip show timesfm

Name: timesfm
Version: 1.2.8
Summary: Open weights time-series foundation model from Google Research.
Home-page: https://github.com/google-research/timesfm
Author: Rajat Sen
Author-email: senrajat@google.com
License: 
Location: /usr/local/lib/python3.11/dist-packages
Editable project location: /content/timesfm
Requires: absl-py, einshape, huggingface_hub, numpy, pandas, scikit-learn, typer, utilsforecast, wandb
Required-by: 


####Imports

In [1]:
from os import path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
import yfinance as yf
from finetuning.finetuning_torch import FinetuningConfig, TimesFMFinetuner
from huggingface_hub import snapshot_download
from torch.utils.data import Dataset

from timesfm import TimesFm, TimesFmCheckpoint, TimesFmHparams
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder
import os

# additional import
from numpy.typing import ArrayLike
import kagglehub

from sklearn.preprocessing import StandardScaler

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.11 (main, Dec  4 2024, 08:55:07) [GCC 11.4.0].


###Dataset Creation

In [2]:
class TimeSeriesDataset(Dataset):
  """Dataset for time series data compatible with TimesFM."""

  def __init__(self,
               series: np.ndarray,
               context_length: int,
               horizon_length: int,
               freq_type: int = 0):
    """
        Initialize dataset.

        Args:
            series: Time series data
            context_length: Number of past timesteps to use as input
            horizon_length: Number of future timesteps to predict
            freq_type: Frequency type (0, 1, or 2)
        """
    if freq_type not in [0, 1, 2]:
      raise ValueError("freq_type must be 0, 1, or 2")

    self.series = series
    self.context_length = context_length
    self.horizon_length = horizon_length
    self.freq_type = freq_type
    self._prepare_samples()

  def _prepare_samples(self) -> None:
    """Prepare sliding window samples from the time series."""
    self.samples = []
    total_length = self.context_length + self.horizon_length

    for start_idx in range(0, len(self.series) - total_length + 1):
      end_idx = start_idx + self.context_length
      x_context = self.series[start_idx:end_idx]
      x_future = self.series[end_idx:end_idx + self.horizon_length]
      self.samples.append((x_context, x_future))

  def __len__(self) -> int:
    return len(self.samples)

  def __getitem__(
      self, index: int
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    x_context, x_future = self.samples[index]

    x_context = torch.tensor(x_context, dtype=torch.float32)
    x_future = torch.tensor(x_future, dtype=torch.float32)

    input_padding = torch.zeros_like(x_context)
    freq = torch.tensor([self.freq_type], dtype=torch.long)

    return x_context, input_padding, freq, x_future

def prepare_datasets(series: np.ndarray,
                     context_length: int,
                     horizon_length: int,
                     freq_type: int = 0,
                     train_split: float = 0.8) -> Tuple[Dataset, Dataset]:
  """
    Prepare training and validation datasets from time series data.

    Args:
        series: Input time series data
        context_length: Number of past timesteps to use
        horizon_length: Number of future timesteps to predict
        freq_type: Frequency type (0, 1, or 2)
        train_split: Fraction of data to use for training

    Returns:
        Tuple of (train_dataset, val_dataset)
    """
  train_size = int(len(series) * train_split)
  train_data = series[:train_size]
  val_data = series[train_size:]

  # Create datasets with specified frequency type
  train_dataset = TimeSeriesDataset(train_data,
                                    context_length=context_length,
                                    horizon_length=horizon_length,
                                    freq_type=freq_type)

  val_dataset = TimeSeriesDataset(val_data,
                                  context_length=context_length,
                                  horizon_length=horizon_length,
                                  freq_type=freq_type)

  return train_dataset, val_dataset

# Modify method to include time_series of your choosing
def get_data(time_series: ArrayLike,
             context_len: int,
             horizon_len: int,
             freq_type: int = 0,
             train_split: float = 0.6) -> Tuple[Dataset, Dataset]:
    """
    Normalize the time series using a StandardScaler (fitted on the training portion)
    and then prepare training and validation datasets.

    Args:
      time_series: 1-D array of time series data.
      context_len: Number of past timesteps to use.
      horizon_len: Number of future timesteps to predict.
      freq_type: Frequency type (0, 1, or 2).
      train_split: Fraction of data to use for training.

    Returns:
      Tuple of (train_dataset, val_dataset)
    """
    # Determine the training split index.
    train_size = int(len(time_series) * train_split)
    train_series = time_series[:train_size].reshape(-1, 1)

    # Fit the StandardScaler on the training data.
    scaler = StandardScaler()
    scaler.fit(train_series)

    # Transform the entire time series.
    normalized_series = scaler.transform(time_series.reshape(-1, 1)).flatten()

    # Create datasets using the normalized series.
    train_dataset, val_dataset = prepare_datasets(
        series=normalized_series,
        context_length=context_len,
        horizon_length=horizon_len,
        freq_type=freq_type,
        train_split=train_split,
    )

    print(f"Created datasets:")
    print(f"- Training samples: {len(train_dataset)}")
    print(f"- Validation samples: {len(val_dataset)}")
    print(f"- Using frequency type: {freq_type}")

    # Optionally, you can return the scaler if you need to invert the normalization later.
    return train_dataset, val_dataset

#### Get data method

In [3]:
import pandas as pd
from numpy.typing import ArrayLike
import kagglehub
import requests
from datetime import datetime, timedelta
from io import StringIO

def get_weather_data(city: str) -> ArrayLike:
    path = kagglehub.dataset_download("gucci1337/weather-of-albania-last-three-years")
    years = [2021, 2022, 2023]
    data_frames = []

    for year in years:
        file_path = f"{path}/data_weather/{city}/{city}{year}.csv"
        df = pd.read_csv(file_path)
        df = df.dropna(subset=['tavg'])  # Remove rows where 'tavg' is NaN
        data_frames.append(df['tavg'])

    # Concatenate the 'tavg' columns from each year's DataFrame
    concatenated_data = pd.concat(data_frames, ignore_index=True)

    return concatenated_data.values

def get_finance_data():
    """
    Returns a numpy array containing the column with the actual data
    """
    CSV_FILE_ABSOLUTE_PATH = "/content/AMZN-stock-price.csv"

    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)

    df = df.iloc[:, 1]

    return df.values



def get_energy_data(year):
    """
    Downloads daily consumption data for a given year, concatenates all days into a single DataFrame,
    and performs basic preprocessing (e.g., dropping NaN values).

    Parameters:
        year (int): The year for which to retrieve data.

    Returns:

    """
    all_data = []
    # Define the start and end dates for the year
    current_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)

    while current_date <= end_date:
        # Format the date as day.month.year (e.g., "1.10.2023" for October 1, 2023)
        # Adjust this format if your URL requires zero-padded day/month values.
        date_str = f"{current_date.day}.{current_date.month}.{current_date.year}"

        # Build the URL by inserting the date string into the appropriate place
        url = f"https://www.eview.de/e1/p3Export.php?frame=StadtMS&p=0005;S~00000936;dg1;t{date_str}"


        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an error for bad status codes

            # Read the CSV data from the response.
            # Here, we assume the CSV file uses ';' as a delimiter.
            daily_df = pd.read_csv(StringIO(response.text), sep=';')
            all_data.append(daily_df)
        except Exception as e:
            print(f"Error fetching data for {date_str}: {e}")

        # Move to the next day
        current_date += timedelta(days=1)

    combined_data = pd.concat(all_data, ignore_index=True)
    string_kws = combined_data.iloc[:, 1].values
    values_float = np.array([float(w.replace(',', '')) for w in string_kws])
    return values_float



def get_healthcare_data():
    """
    Returns a numpy array containing the column with the actual data
    """
    CSV_FILE_ABSOLUTE_PATH = "/content/WHO-COVID-19-global-daily-data.csv"

    df = pd.read_csv(CSV_FILE_ABSOLUTE_PATH)
    df = df.sort_values(['Country', 'Date_reported'])
    df['New_cases'] = df.groupby('Country')['New_cases'].transform(lambda group: group.interpolate(method='linear'))

    df = get_new_cases_by_country(df)

    new_cases = df["Germany"][200: 1200]

    return new_cases

def get_new_cases_by_country(df: pd.DataFrame) -> dict:
    """
    Given a DataFrame with columns:
      'Date_reported', 'Country_code', 'Country', 'WHO_region',
      'New_cases', 'Cumulative_cases', 'New_deaths', 'Cumulative_deaths'
    returns a dictionary where:
      - keys are country names (as strings)
      - values are NumPy arrays of new cases (from the 'New_cases' column)

    The function sorts the data by 'Date_reported' for each country to ensure
    a consistent time-series order.
    """
    result = {}
    # Group the data by the 'Country' column
    for country, group in df.groupby('Country'):
        # Sort the group's rows by the reported date
        group_sorted = group.sort_values('Date_reported')
        # Extract the 'New_cases' column as a NumPy array
        new_cases_array = group_sorted['New_cases'].to_numpy()
        result[country] = new_cases_array
    return result

###Model Creation

In [4]:
from os import path
import torch
from huggingface_hub import snapshot_download
# Make sure to import ResidualBlock from the appropriate module.
from timesfm.pytorch_patched_decoder import ResidualBlock


def get_model(load_weights: bool,
              per_core_batch_size: int,
              horizon_len: int,
              context_len: int):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    repo_id = "google/timesfm-2.0-500m-pytorch"

    hparams = TimesFmHparams(
        backend=device,
        per_core_batch_size=per_core_batch_size,
        horizon_len=horizon_len,  # desired horizon length
        num_layers=50,
        use_positional_embedding=False,
        context_len=context_len)

    tfm = TimesFm(hparams=hparams,
                  checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id))

    model = PatchedTimeSeriesDecoder(tfm._model_config)
    if load_weights:
        checkpoint_path = path.join(snapshot_download(repo_id), "torch_model.ckpt")
        loaded_checkpoint = torch.load(checkpoint_path, weights_only=True)
        model.load_state_dict(loaded_checkpoint)

        # Retrieve pre-trained horizon length and compute output dimensions.
        pretrained_horizon = tfm._model_config.horizon_len
        num_quantiles = len(tfm._model_config.quantiles)
        pretrained_output_dims = pretrained_horizon * (1 + num_quantiles)
        new_output_dims = horizon_len * (1 + num_quantiles)

        if new_output_dims != pretrained_output_dims:
            print(f"Pre-trained horizon length: {pretrained_horizon} (output dims {pretrained_output_dims}) vs. desired horizon length: {horizon_len} (output dims {new_output_dims}).")

            old_layer = model.horizon_ff_layer  # This is a ResidualBlock.
            hidden_size = tfm._model_config.hidden_size
            intermediate_size = tfm._model_config.intermediate_size

            # Create a new final layer with the new output dimensions.
            new_layer = ResidualBlock(input_dims=hidden_size,
                                      output_dims=new_output_dims,
                                      hidden_dims=intermediate_size)

            # Reuse weights from old_layer by slicing the first new_output_dims rows.
            with torch.no_grad():
                # For the output_layer.
                new_layer.output_layer.weight[:new_output_dims, :] = \
                    old_layer.output_layer.weight[:new_output_dims, :]
                new_layer.output_layer.bias[:new_output_dims] = \
                    old_layer.output_layer.bias[:new_output_dims]

                # For the residual_layer.
                new_layer.residual_layer.weight[:new_output_dims, :] = \
                    old_layer.residual_layer.weight[:new_output_dims, :]
                new_layer.residual_layer.bias[:new_output_dims] = \
                    old_layer.residual_layer.bias[:new_output_dims]

            # Replace the model's final layer with the new one.
            model.horizon_ff_layer = new_layer
            # Update the configuration to reflect the new horizon length.
            tfm._model_config.horizon_len = horizon_len
    return model, hparams, tfm._model_config


###Plotting Method

In [5]:
def plot_predictions(
    model: TimesFm,
    val_dataset: Dataset,
    save_path: Optional[str] = "predictions.png",
) -> None:
  """
    Plot model predictions against ground truth for a batch of validation data.

    Args:
      model: Trained TimesFM model
      val_dataset: Validation dataset
      save_path: Path to save the plot
    """
  import matplotlib.pyplot as plt

  model.eval()

  x_context, x_padding, freq, x_future = val_dataset[0]
  x_context = x_context.unsqueeze(0)  # Add batch dimension
  x_padding = x_padding.unsqueeze(0)
  freq = freq.unsqueeze(0)
  x_future = x_future.unsqueeze(0)

  device = next(model.parameters()).device
  x_context = x_context.to(device)
  x_padding = x_padding.to(device)
  freq = freq.to(device)
  x_future = x_future.to(device)

  with torch.no_grad():
    predictions = model(x_context, x_padding.float(), freq)
    predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
    last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

  context_vals = x_context[0].cpu().numpy()
  future_vals = x_future[0].cpu().numpy()
  pred_vals = last_patch_pred[0].cpu().numpy()

  context_len = len(context_vals)
  horizon_len = len(future_vals)

  plt.figure(figsize=(12, 6))

  plt.plot(range(context_len),
           context_vals,
           label="Historical Data",
           color="blue",
           linewidth=2)

  plt.plot(
      range(context_len, context_len + horizon_len),
      future_vals,
      label="Ground Truth",
      color="green",
      linestyle="--",
      linewidth=2,
  )

  plt.plot(range(context_len, context_len + horizon_len),
           pred_vals,
           label="Prediction",
           color="red",
           linewidth=2)

  plt.xlabel("Time Step")
  plt.ylabel("Value")
  plt.title("TimesFM Predictions vs Ground Truth")
  plt.legend()
  plt.grid(True)

  if save_path:
    plt.savefig(save_path)
    print(f"Plot saved to {save_path}")

  plt.close()

###Evaluation Method

In [6]:
import numpy as np
import torch

def evaluate_model_mae(model, val_dataset) -> float:
    """
    Evaluate the TimesFM model on the validation dataset using Mean Absolute Error (MAE).

    Args:
      model: Trained TimesFM model.
      val_dataset: Validation dataset that returns tuples of
                   (x_context, x_padding, freq, x_future) for each sample.

    Returns:
      The average MAE across all samples in the validation dataset.
    """
    model.eval()
    device = next(model.parameters()).device
    mae_list = []

    # Iterate over each sample in the validation dataset
    for sample in val_dataset:
        # Unpack the sample: context, padding, frequency indicator, and future ground truth.
        x_context, x_padding, freq, x_future = sample

        # Add a batch dimension (model expects batched input)
        x_context = x_context.unsqueeze(0).to(device)
        x_padding = x_padding.unsqueeze(0).to(device)
        freq = freq.unsqueeze(0).to(device)
        x_future = x_future.unsqueeze(0).to(device)

        # Get the prediction without tracking gradients
        with torch.no_grad():
            predictions = model(x_context, x_padding.float(), freq)
            # Assume output shape is [B, N, horizon_len, ...]
            predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
            last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

        # Convert predictions and ground truth to NumPy arrays
        pred_vals = last_patch_pred[0].cpu().numpy()
        true_vals = x_future[0].cpu().numpy()

        # Compute the MAE for this sample
        sample_mae = np.mean(np.abs(pred_vals - true_vals))
        mae_list.append(sample_mae)

    # Return the average MAE over all samples
    return np.mean(mae_list)


def evaluate_model_rmse(model, val_dataset) -> float:
    """
    Evaluate the TimesFM model on the validation dataset using Root Mean Squared Error (RMSE).

    Args:
      model: Trained TimesFM model.
      val_dataset: Validation dataset that returns tuples of
                   (x_context, x_padding, freq, x_future) for each sample.

    Returns:
      The average RMSE across all samples in the validation dataset.
    """
    model.eval()
    device = next(model.parameters()).device
    rmse_list = []

    for sample in val_dataset:
        x_context, x_padding, freq, x_future = sample

        x_context = x_context.unsqueeze(0).to(device)
        x_padding = x_padding.unsqueeze(0).to(device)
        freq = freq.unsqueeze(0).to(device)
        x_future = x_future.unsqueeze(0).to(device)

        with torch.no_grad():
            predictions = model(x_context, x_padding.float(), freq)
            predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
            last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

        pred_vals = last_patch_pred[0].cpu().numpy()
        true_vals = x_future[0].cpu().numpy()

        sample_rmse = np.sqrt(np.mean((pred_vals - true_vals) ** 2))
        rmse_list.append(sample_rmse)

    return np.mean(rmse_list)


def evaluate_model_mase(model, val_dataset, insample_data) -> float:
    """
    Evaluate the TimesFM model on the validation dataset using Mean Absolute Scaled Error (MASE).

    Args:
      model: Trained TimesFM model.
      val_dataset: Validation dataset that returns tuples of
                   (x_context, x_padding, freq, x_future) for each sample.
      insample_data: In-sample data (historical values) as a 1-D NumPy array
                     used to compute the naive forecast scale.

    Returns:
      The average MASE across all samples in the validation dataset.
    """
    model.eval()
    device = next(model.parameters()).device
    mase_list = []

    # Compute naive forecast error scale from the entire in-sample data.
    insample_vals = np.array(insample_data)  # Expecting a 1-D array
    naive_forecast_errors = np.abs(insample_vals[1:] - insample_vals[:-1])
    scale = np.mean(naive_forecast_errors) if naive_forecast_errors.size > 0 else 1e-8

    for sample in val_dataset:
        x_context, x_padding, freq, x_future = sample

        x_context = x_context.unsqueeze(0).to(device)
        x_padding = x_padding.unsqueeze(0).to(device)
        freq = freq.unsqueeze(0).to(device)
        x_future = x_future.unsqueeze(0).to(device)

        with torch.no_grad():
            predictions = model(x_context, x_padding.float(), freq)
            predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
            last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

        pred_vals = last_patch_pred[0].cpu().numpy()
        true_vals = x_future[0].cpu().numpy()

        sample_mase = np.mean(np.abs(pred_vals - true_vals)) / scale
        mase_list.append(sample_mase)

    return np.mean(mase_list)


###Fine-tuning on Single GPU

In [7]:
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score, explained_variance_score
import gc

# Assume that the following functions and classes are defined and imported:
#   get_model, get_data, TimesFMFinetuner, FinetuningConfig,
#   evaluate_model_mae, evaluate_model_mase, evaluate_model_rmse, plot_predictions
#
# Also assume that the data loading functions are defined:
#   get_weather_data(city), get_finance_data(), get_energy_data(year), get_healthcare_data()

# Define arrays for context (sequence) lengths and forecast (horizon) lengths.
context_lengths = [64, 128, 256]
horizon_lengths = [32, 64, 128]

# The train/validation split fraction (e.g., first 60% for training)
train_split = 0.6

def run_experiment(dataset_name: str, time_series: np.ndarray):
    print(f"\n===== Dataset: {dataset_name} =====")

    # Loop over each experiment configuration (using fixed context and horizon lengths).
    for i in range(len(context_lengths)):
        context_len = context_lengths[i]
        horizon_len = horizon_lengths[i]

        print(f"\n--- Experiment {i+1} with context_len={context_len}, horizon_len={horizon_len} ---")

        # Define an objective function for Optuna tuning.
        def objective(trial: optuna.Trial):
            # Suggest a learning rate (tune within a logarithmic scale).
            learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
            # Suggest a batch size from a set of candidates.
            batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

            # Get the model (model loading remains unchanged).
            model, hparams, tfm_config = get_model(load_weights=True,
                                                   per_core_batch_size=batch_size,
                                                   horizon_len=horizon_len,
                                                   context_len=context_len)

            # Set up finetuning configuration with the trial's learning rate and batch size.
            config = FinetuningConfig(batch_size=batch_size,
                                      num_epochs=1,  # Use a short run for tuning; adjust as needed.
                                      learning_rate=learning_rate,
                                      use_wandb=False,
                                      freq_type=1,
                                      log_every_n_steps=10,
                                      val_check_interval=0.5,
                                      use_quantile_loss=True)

            # Create the train and validation datasets.
            train_dataset, val_dataset = get_data(time_series,
                                                  context_len=context_len,
                                                  horizon_len=horizon_len,
                                                  freq_type=config.freq_type,
                                                  train_split=train_split)

            # The finetuner expects datasets; they are not wrapped in a DataLoader.
            finetuner = TimesFMFinetuner(model, config)
            finetuner.finetune(train_dataset=train_dataset, val_dataset=val_dataset)

            # Evaluate the model; here we choose RMSE as the metric to minimize.
            rmse = evaluate_model_rmse(model, val_dataset)
            print(f"Trial {trial.number}: learning_rate={learning_rate:.6f}, batch_size={batch_size}, RMSE={rmse:.4f}")
            return rmse

        # Create an Optuna study for this experiment.
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=5)  # Adjust n_trials as needed for your tuning

        best_params = study.best_trial.params
        best_lr = best_params["learning_rate"]
        best_batch_size = best_params["batch_size"]
        print(f"\nBest parameters for experiment {i+1}: learning_rate={best_lr:.6f}, batch_size={best_batch_size}")

        # Now, retrain the model using the best hyperparameters.
        model, hparams, tfm_config = get_model(load_weights=True,
                                               per_core_batch_size=best_batch_size,
                                               horizon_len=horizon_len,
                                               context_len=context_len)
        config = FinetuningConfig(batch_size=best_batch_size,
                                  num_epochs=100,  # Adjust to a full training run if desired.
                                  learning_rate=best_lr,
                                  use_wandb=False,
                                  freq_type=1,
                                  log_every_n_steps=10,
                                  val_check_interval=0.5,
                                  use_quantile_loss=True)
        train_dataset, val_dataset = get_data(time_series,
                                              context_len=context_len,
                                              horizon_len=horizon_len,
                                              freq_type=config.freq_type,
                                              train_split=train_split)
        finetuner = TimesFMFinetuner(model, config)
        print(f"\nStarting final finetuning for {dataset_name} experiment {i+1} using best parameters: learning_rate={best_lr:.6f}, batch_size={best_batch_size}...")
        results = finetuner.finetune(train_dataset=train_dataset, val_dataset=val_dataset)
        print("Finetuning completed!")
        print(f"Training history: {len(results['history']['train_loss'])} epochs")

        # --- Final Evaluation ---
        # Existing evaluation metrics
        mae_custom = evaluate_model_mae(model, val_dataset)
        mase_score = evaluate_model_mase(model, val_dataset, time_series[:int(len(time_series) * train_split)])
        rmse_custom = evaluate_model_rmse(model, val_dataset)
        print(f"Custom Evaluation -> MAE: {mae_custom:.4f}, MASE: {mase_score:.4f}, RMSE: {rmse_custom:.4f}")

        # Now, compute additional evaluation metrics using sklearn.
        model.eval()
        all_preds = []
        all_targets = []

        # Get model's device (e.g., cuda:0 or cpu)
        device = next(model.parameters()).device

        with torch.no_grad():
            for sample in val_dataset:
                # Unpack the sample: (x_context, input_padding, freq, x_future)
                x_context, input_padding, freq, x_future = sample

                # Add a batch dimension if missing and move to the model's device.
                if x_context.dim() == 1:
                    x_context = x_context.unsqueeze(0).to(device)
                    input_padding = input_padding.unsqueeze(0).to(device)
                    freq = freq.unsqueeze(0).to(device)
                    x_future = x_future.unsqueeze(0).to(device)
                else:
                    x_context = x_context.to(device)
                    input_padding = input_padding.to(device)
                    freq = freq.to(device)
                    x_future = x_future.to(device)

                with torch.no_grad():
                  predictions = model(x_context, input_padding.float(), freq)
                  predictions_mean = predictions[..., 0]  # select the first output channel if applicable
                  last_patch_pred = predictions_mean[:, -1, :]  # select the last patch's prediction
                  preds = last_patch_pred.squeeze().cpu().flatten()
                  target = x_future.squeeze().cpu().flatten()

                  all_preds.append(preds)
                  all_targets.append(target)

        # Concatenate all predictions and targets into 1D arrays.
        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        # Compute error-based metrics.
        rmse_sklearn = np.sqrt(mean_squared_error(all_targets, all_preds))
        mae_sklearn = mean_absolute_error(all_targets, all_preds)
        mape_sklearn = mean_absolute_percentage_error(all_targets, all_preds)

        # Compute distribution-capturing metrics.
        r2 = r2_score(all_targets, all_preds)
        explained_var = explained_variance_score(all_targets, all_preds)

        print("\nSklearn Evaluation Metrics:")
        print(f"RMSE: {rmse_sklearn:.4f}")
        print(f"MAE: {mae_sklearn:.4f}")
        print(f"MAPE: {mape_sklearn:.4f}")
        print(f"Coefficient of Determination (R^2): {r2:.4f}")
        print(f"Explained Variance: {explained_var:.4f}")

        # Optionally, plot predictions from the validation set.
        plot_predictions(model=model,
                         val_dataset=val_dataset,
                         save_path=f"{dataset_name}_predictions_exp{i+1}.png")

        print("-" * 60)
        # Free GPU memory: delete objects, run garbage collection, and empty CUDA cache.
        del model, train_dataset, val_dataset, finetuner, results, best_params
        gc.collect()
        torch.cuda.empty_cache()


# Run experiments for each dataset:
# 1. Weather Data (using city "lezhe")
city = "lezhe"
#weather_series = get_weather_data(city)
#run_experiment("weather", weather_series)

In [8]:
# 2. Energy Data (using year 2024)
year = 2024
#energy_series = get_energy_data(year)
#run_experiment("energy", energy_series)

In [10]:
# 3. Finance Data
finance_series = get_finance_data()
run_experiment("finance", finance_series)

[I 2025-03-04 08:01:46,391] A new study created in memory with name: no-name-b9f253ee-3fcd-4de5-b2fe-ad6f52822012



===== Dataset: finance =====

--- Experiment 1 with context_len=64, horizon_len=32 ---


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

...

[I 2025-03-04 09:28:29,131] Trial 4 finished with value: 12.807668685913086 and parameters: {'learning_rate': 0.0009570336958280227, 'batch_size': 32}. Best is trial 3 with value: 2.433497190475464.


Trial 4: learning_rate=0.000957, batch_size=32, RMSE=12.8077

Best parameters for experiment 3: learning_rate=0.000161, batch_size=32


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Created datasets:
- Training samples: 2669
- Validation samples: 1653
- Using frequency type: 1

Starting final finetuning for finance experiment 3 using best parameters: learning_rate=0.000161, batch_size=32...
Finetuning completed!
Training history: 100 epochs
Custom Evaluation -> MAE: 2.2843, MASE: 1.9120, RMSE: 2.6323

Sklearn Evaluation Metrics:
RMSE: 3.5310
MAE: 2.2843
MAPE: 0.1712
Coefficient of Determination (R^2): 0.8332
Explained Variance: 0.8336
Plot saved to finance_predictions_exp3.png
------------------------------------------------------------


In [11]:
# 4. Healthcare Data
healthcare_series = get_healthcare_data()
run_experiment("healthcare", healthcare_series)

[I 2025-03-04 10:39:36,109] A new study created in memory with name: no-name-1af751cf-5fc1-4295-acd6-1d7389184718



===== Dataset: healthcare =====

--- Experiment 1 with context_len=64, horizon_len=32 ---


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Pre-trained horizon length: 128 (output dims 1280) vs. desired horizon length: 32 (output dims 320).
Created datasets:
- Training samples: 505
- Validation samples: 305
- Using frequency type: 1


...

[I 2025-03-04 10:57:37,523] Trial 4 finished with value: 2.807001829147339 and parameters: {'learning_rate': 3.0503038901768396e-05, 'batch_size': 128}. Best is trial 4 with value: 2.807001829147339.


Trial 4: learning_rate=0.000031, batch_size=128, RMSE=2.8070

Best parameters for experiment 3: learning_rate=0.000031, batch_size=128


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Created datasets:
- Training samples: 217
- Validation samples: 17
- Using frequency type: 1

Starting final finetuning for healthcare experiment 3 using best parameters: learning_rate=0.000031, batch_size=128...
Finetuning completed!
Training history: 100 epochs
Custom Evaluation -> MAE: 5.9844, MASE: 0.0012, RMSE: 7.5309

Sklearn Evaluation Metrics:
RMSE: 7.5513
MAE: 5.9844
MAPE: 23.6367
Coefficient of Determination (R^2): -1453.4497
Explained Variance: -539.9816
Plot saved to healthcare_predictions_exp3.png
------------------------------------------------------------


###Show the directory where the graph is created

In [12]:
!ls

AMZN-stock-price.csv	      healthcare_predictions_exp1.png  timesfm
finance_predictions_exp1.png  healthcare_predictions_exp2.png  WHO-COVID-19-global-daily-data.csv
finance_predictions_exp2.png  healthcare_predictions_exp3.png
finance_predictions_exp3.png  sample_data


###Freeing Up GPU Resources

In [13]:
from numba import cuda
device = cuda.get_current_device()
device.reset()
device = cuda.get_current_device()